In [ ]:
# Copyright 2025 DeepMind Technologies Limited. All Rights Reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

[![Colab で開く](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-gemini/genai-processors/blob/main/notebooks/content_api_intro.ipynb)


# GenAI Processors Content API へようこそ

このノートブックは、GenAI Processors ライブラリ内の `content_api` モジュールの入門編です。`ProcessorPart` と `ProcessorContent` といったコンテンツの基本構成要素の作成・操作・連携方法を学びます。

【学べること】

* 文字列、bytes、PIL 画像、GenAI の Part などから `ProcessorPart` を作成する方法
* `ProcessorPart` の主要属性（`mimetype`、`substream_name`、`role`、`metadata` など）
* 複数の `ProcessorPart` をまとめる `ProcessorContent` の構築方法
* コンテンツ操作に役立つユーティリティ関数（例: `as_text`）
* `ProcessorPart` と `ProcessorContent` を GenAI モデルと統合する方法

それでは、AI パイプラインにおける構造化コンテンツの力を体験していきましょう！


## 1. 🛠️ セットアップ


In [ ]:
!pip install genai_processors

In [ ]:
# @title Import modules
import dataclasses
import dataclasses_json
from genai_processors import content_api
from genai_processors import processor
from genai_processors import streams
from genai_processors.core import genai_model
from google.colab import userdata
from google.genai import types as genai_types
from IPython.display import display
import nest_asyncio
import PIL.Image
import requests

nest_asyncio.apply()  # Needed to run async loops in Colab

# Convenient aliases
ProcessorPart = content_api.ProcessorPart
ProcessorContent = content_api.ProcessorContent
as_text = content_api.as_text
is_text = content_api.is_text
is_json = content_api.is_json

# For GenAI Model interaction (optional, but useful for demonstration!)
try:
  API_KEY = userdata.get("GOOGLE_API_KEY")
  if not API_KEY:
    print(
        "⚠️ API Key not found in Colab secrets. GenAI model examples will be"
        " skipped."
    )
    print(
        "To run these, add your Gemini API Key to Colab Secrets with the name"
        " 'GOOGLE_API_KEY'."
    )
except userdata.SecretNotFoundError:
  API_KEY = None  # Or set your API key directly here if not using Colab secrets
  print(
      "`userdata` not imported. Set API_KEY manually if you want to run GenAI"
      " model examples."
  )

GenaiModel = genai_model.GenaiModel


def generate_gdm_logo_bytes() -> bytes:
  # The URL of the DeepMind logo image
  image_url = "https://upload.wikimedia.org/wikipedia/commons/thumb/6/6a/DeepMind_new_logo.svg/2560px-DeepMind_new_logo.svg.png"
  headers = {"User-Agent": "genai_processors Colab"}
  response = requests.get(image_url, headers=headers)
  response.raise_for_status()
  return response.content

## 2. 🧱 ProcessorPart: コンテンツの最小単位

`ProcessorPart` は、プロセッサが扱う最小かつ不可分のコンテンツ片です。単一のモダリティと単一のロールを持つ型付きコンテナ、と考えるとわかりやすいでしょう。

`ProcessorPart` はさまざまな Python 型から作成できます:


In [ ]:
text_data = "Hello, GenAI World! This is a text part."
text_part = ProcessorPart(text_data)

print(f"Original data: {text_data}")
print(f"ProcessorPart: {text_part}")
print(
    f"MIME type: {text_part.mimetype}"
)  # Automatically inferred as 'text/plain'
print(f"Text content: '{text_part.text}'")

#### ProcessorPart の主な属性

*   `part`: 基盤となる `google.genai.types.Part` オブジェクト。
*   `text`: テキストコンテンツへアクセスします（テキストでない場合は ValueError）。
*   `bytes`: コンテンツをバイト列として取得します。
*   `pil_image`: コンテンツを PIL 画像として取得します（画像でない場合は ValueError）。
*   `mimetype`: コンテンツの [MIME タイプ](https://developer.mozilla.org/en-US/docs/Web/HTTP/Basics_of_HTTP/MIME_types)（例: `text/plain`, `image/png`, `application/json`）。
*   `role`: コンテンツの作成者（例: "user", "model", "tool"）。会話 AI で有用。デフォルトは 空文字列。
*   `substream_name`: パートの分類やルーティングに用いる任意の文字列。異なる情報タイプや代替応答を区別するのに便利。デフォルトは 空文字列。
    - いくつかのサブストリーム名は特別な意味を持ちます。例えば `status` と `debug` はユーザーへ早期に返す必要がある内容用で、パイプラインの後段では処理されません。
    - `realtime` は Live API の `send_realtime_content()`（`send_client_content()` と対比）を用いるコンテンツ用です。
*   `metadata`: 任意の補足情報を格納する辞書。

例として、より多くの属性を付与したテキストパートを作成してみます:


In [ ]:
detailed_text_part = ProcessorPart(
    "This is a user query.",
    role="user",
    substream_name="user_query_main",
    metadata={
        "timestamp": "2024-07-29T10:00:00Z",
        "session_id": "xyz123",
    },
)

print(f"ProcessorPart: {detailed_text_part}")
print(f"Role: {detailed_text_part.role}")
print(f"Substream Name: {detailed_text_part.substream_name}")
print(f"Custom Metadata: {detailed_text_part.metadata}")
print(
    f"Timestamp from metadata: {detailed_text_part.get_metadata('timestamp')}"
)

### Bytes から作成する（例: 画像データ）

生のバイト列から `ProcessorPart` を作成する場合は、`mimetype` を必ず指定する必要があります。


In [ ]:
gdm_png_bytes = generate_gdm_logo_bytes()

image_bytes_part = ProcessorPart(gdm_png_bytes, mimetype="image/png")

print(f"ProcessorPart (from bytes): {image_bytes_part}")
print(f"MIME type: {image_bytes_part.mimetype}")
print(f"Has bytes: {image_bytes_part.bytes is not None}")

# You can access it as a PIL Image too
try:
  pil_img = image_bytes_part.pil_image
  display(pil_img)
except Exception as e:
  print(f"Error converting to PIL Image: {e}")

### PIL（Pillow）の Image オブジェクトから作成

`PIL.Image.Image` オブジェクトをそのまま渡せます。ライブラリがバイト列への変換と MIME タイプの推論を行います（PIL 画像にフォーマットがない場合は `image/webp` を既定、フォーマットがあれば `image/png` や `image/jpeg` など既存のフォーマットを使用）。


In [ ]:
# Create a simple PIL Image
pil_image_obj = PIL.Image.new("RGB", (60, 30), color="red")
pil_image_part = ProcessorPart(pil_image_obj)

print(f"ProcessorPart (from PIL Image): {pil_image_part}")
print(
    f"Inferred MIME type: {pil_image_part.mimetype}"
)  # Likely image/webp or image/png
display(pil_image_part.pil_image)  # Display in Colab

# You can also specify a mimetype if you want a different format
jpeg_pil_image_part = ProcessorPart(pil_image_obj, mimetype="image/jpeg")
print(f"ProcessorPart (from PIL with specified JPEG): {jpeg_pil_image_part}")
print(f"MIME type: {jpeg_pil_image_part.mimetype}")
display(jpeg_pil_image_part.pil_image)

### `google.genai.types.Part` から作成

Google AI SDK を利用している場合、`genai_types.Part` オブジェクトを `ProcessorPart` に直接ラップできます。


In [ ]:
genai_text_part_sdk = genai_types.Part(text="From GenAI SDK!")
processor_part_from_sdk = ProcessorPart(genai_text_part_sdk, role="model")

print(f"ProcessorPart (from GenAI SDK Part): {processor_part_from_sdk}")
print(f"Text: {processor_part_from_sdk.text}")
print(f"Role: {processor_part_from_sdk.role}")

### 既存の ProcessorPart から作成

既存の `ProcessorPart` から新しい `ProcessorPart` を作成すると、新しいインスタンスが作られ、既存の `ProcessorPart` が持つ基盤の `genai_types.Part` を使います。つまり、元の `ProcessorPart` の基盤 `genai_types.Part` に対する変更は新しいインスタンスにも反映されます。

この方法で作成する際、`role`、`substream_name`、`metadata` といった属性を上書きできます。


In [ ]:
original_part = ProcessorPart(
    "Original message.",
    role="user",
    substream_name="original_stream",
    metadata={"version": 1},
)

# Simple copy
copied_part = ProcessorPart(original_part)
print(f"Original: {original_part}")
print(f"Copied:   {copied_part}")
print(f"Are they the same object? {original_part is copied_part}")  # False
print(f"Are they equal in value? {original_part == copied_part}")  # True

# Copy with overridden attributes
modified_copy_part = ProcessorPart(
    original_part,
    role="model",  # Changed role
    substream_name="modified_stream",  # Changed substream
    metadata={"version": 2, "status": "processed"},  # New metadata
)
print(f"\nModified Copy: {modified_copy_part}")
print(f"Role: {modified_copy_part.role}")
print(f"Substream: {modified_copy_part.substream_name}")
print(f"Metadata: {modified_copy_part.metadata}")

### Python の Dataclass（構造化データ）から作成

構造化データには `ProcessorPart.from_dataclass()` を利用できます。これにより dataclass が JSON にシリアライズされ、`mimetype` は `application/json; type=<ClassName>` に設定されます。


In [ ]:
@dataclasses_json.dataclass_json  # Important for JSON serialization
@dataclasses.dataclass
class MyStructuredData:
  id: int
  name: str
  tags: list[str]


my_data_instance = MyStructuredData(
    id=101, name="Alpha", tags=["important", "beta"]
)

dataclass_part = ProcessorPart.from_dataclass(
    dataclass=my_data_instance,
    role="system_event",
    metadata={"source": "internal_module"},
)

print(f"ProcessorPart (from dataclass): {dataclass_part}")
print(f"MIME type: {dataclass_part.mimetype}")
print(f"Underlying text (JSON): {dataclass_part.text}")
assert is_json(dataclass_part.mimetype)

# To get the dataclass back:
retrieved_data_instance = dataclass_part.get_dataclass(MyStructuredData)
print(f"\nRetrieved dataclass instance: {retrieved_data_instance}")
print(
    "Is original equal to retrieved?"
    f" {my_data_instance == retrieved_data_instance}"
)

## 3. 📦 ProcessorContent: パートの集合

`ProcessorContent` は 1 つ以上の `ProcessorPart` を格納するコンテナです。会話の 1 ターン全体や、マルチモーダル入力の集合を表すのによく使われます。

`ProcessorPart` のリストのように振る舞い、コンストラクタに `ProcessorPart`（あるいは `ProcessorPart` に変換可能なデータ）を渡して構築します。


### ProcessorContent の作成


In [ ]:
# From individual strings/parts
content1 = ProcessorContent(
    "This is the first part.",
    ProcessorPart(generate_gdm_logo_bytes(), mimetype="image/png", role="user"),
    "And a final textual comment.",
)

print("Content 1:")
for part in content1:  # You can iterate directly over ProcessorContent
  print(
      f"  - {part.mimetype}:"
      f" {part.text if is_text(part.mimetype) else '[binary data]'}"
  )
print(f"Length of Content 1: {len(content1)}")

# From a list of ProcessorPart objects
parts_list = [
    ProcessorPart("Query about cats.", role="user"),
    ProcessorPart("Cats are fascinating creatures!", role="model"),
]
content2 = ProcessorContent(parts_list)

print("\nContent 2:")
for (
    mime,
    part_obj,
) in content2.items():  # .items() yields (mimetype, ProcessorPart)
  print(f"  - Role: {part_obj.role}, Mimetype: {mime}, Text: {part_obj.text}")

# From another ProcessorContent object (creates a new collection)
content3 = ProcessorContent(content1)
print(f"\nContent 3 (copy of Content 1):")
print(f"Is Content 1 same object as Content 3? {content1 is content3}")
print(f"Is Content 1 equal to Content 3? {content1 == content3}")

### ユーティリティ: `as_text()`

`ProcessorContent` からテキスト情報だけを取り出したいケースはよくあります。`content_api.as_text()` はまさにそれを行い、テキスト系パートの文字列を連結して返します。`substream_name` 引数を指定すれば、特定サブストリームのテキストを抽出可能です。


In [ ]:
multimodal_content = ProcessorContent(
    ProcessorPart("Here is some initial text. ", substream_name="other"),
    ProcessorPart(
        generate_gdm_logo_bytes(), mimetype="image/png"
    ),  # This will be ignored by as_text
    "Followed by more text. ",
)

all_text = as_text(multimodal_content)
print(f"Concatenated text from multimodal_content:\n{all_text}")

other_text_only = as_text(multimodal_content, substream_name="other")
print(f"Text from 'other' substream:\n{other_text_only}")

## 4. 🤖 GenAI モデルとの統合（任意）

`ProcessorPart` と `ProcessorContent` は、`GenaiModel` プロセッサを通じて GenAI モデルとシームレスに連携します。`GenaiModel` は `AsyncIterable[ProcessorPart]` を入力として期待しており、`ProcessorContent`（や `ProcessorPart` のリスト、さらには文字列）も `streams.stream_content()` を使って簡単にそのようなストリームへ変換できます。


#### 非同期（Async）


In [ ]:
p_genai = GenaiModel(
    api_key=API_KEY,
    model_name="gemini-2.0-flash-lite",
)
input_stream = streams.stream_content(
    "What is the best part of owning a Dalmatian?"
)
async for content_part in p_genai(input_stream):
  print(content_part.text)

#### 同期（Sync）

同期環境でモデル呼び出しがブロック動作の方が好ましい場合は、`apply_sync` メソッドを利用してください。これは `ProcessorPart` のリストを入力として受け取ります。


In [ ]:
p = GenaiModel(
    api_key=API_KEY,
    model_name='gemini-2.0-flash-lite',
)

genai_content = processor.apply_sync(
    p, ['What is the best part of owning a Dalmatian?']
)

print('Raw parts\n\n')
for content_part in genai_content:
  print(content_part)
print('\n\n')
print('Using `content_api.as_text:\n\n')
print(content_api.as_text(genai_content))

## 5. 次のステップ

このチュートリアルでは、`ProcessorPart` と `ProcessorContent` の作成方法、そしてそれらが GenAI Processors におけるコンテンツ処理の基礎となることを学びました。また、GenAI モデルとの統合方法も確認しました。

続けて学ぶには、
[processor intro](https://colab.research.google.com/github/google-gemini/genai-processors/blob/main/notebooks/processor_intro.ipynb)
のノートブックで、Live API を用いたリアルタイムプロセッサの作成に踏み込んでみてください。
